# Create embeddings of documents

#### Setup Environment

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd
from semantic_text_splitter import TextSplitter
from tokenizers import Tokenizer

In [2]:
# Loads variables from the environment
load_dotenv()
open_api_key = os.getenv("OPENAI_API_KEY")

#### Read in and Chunk Documents 

#### Create Embedding Vectors for Chunks

In [4]:
client = OpenAI(api_key=open_api_key)

max_tokens = 1023 # 8191 is max length for text-embedding-3-large
tokenizer = Tokenizer.from_pretrained("bert-base-uncased")
splitter = TextSplitter.from_huggingface_tokenizer(tokenizer, max_tokens)

# Create lists to store the data
embeddings = []
video_ids = []
chunk_indices = []
chunk_texts = []

# ToDo: add some sort of loop through all videos
documents = ['E7W4OQfJWdw', 'cS7cNaBrkxo']
for video_id in documents:

    with open(f'data/documents/{video_id}.txt', 'r', encoding='utf-8') as file:
        text_content = file.read()

    chunks = splitter.chunks(text_content)

    for chunk in chunks:
        # Create chunks directory if it doesn't exist
        os.makedirs('data/chunks', exist_ok=True)
        
        # Write chunk to file with video ID and chunk number
        chunk_filename = f'data/chunks/{video_id}_chunk{chunks.index(chunk)}.txt'
        with open(chunk_filename, 'w', encoding='utf-8') as f:
            f.write(chunk)
        response = client.embeddings.create(
            input=chunk,
            model="text-embedding-3-small"
        )

        embeddings.append(response.data[0].embedding)
        video_ids.append(video_id)
        chunk_indices.append(chunks.index(chunk))
        # chunk_texts.append(chunk)


In [5]:
# Create DataFrame
df = pd.DataFrame({
    'embedding': embeddings,
    'video_id': video_ids, 
    'chunk_index': chunk_indices,
    # 'chunk_text': chunk_texts
})

In [6]:
df.head()

,embedding,video_id,chunk_index
0,"[0.010170252993702888, -0.015520867891609669, ...",E7W4OQfJWdw,0
1,"[0.00484744505956769, 0.008455636911094189, -0...",E7W4OQfJWdw,1
2,"[0.017064958810806274, -0.0008320452179759741,...",E7W4OQfJWdw,2
3,"[0.018084809184074402, -0.010019644163548946, ...",E7W4OQfJWdw,3
4,"[0.046452846378088, -0.005249288398772478, -0....",E7W4OQfJWdw,4


In [7]:
# Save DataFrame to CSV
df.to_csv('data/embeddings.csv', index=False)